# Train the plan segmenter on a Colab GPU

Trains the wall + room U-Net from [`plan-to-cost`](https://github.com/Tgedin/buildable-land-map/tree/main/plan-to-cost) on CubiCasa5k, then scores it **once** on the test split.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU` (or any GPU). Then run the cells in order.

**Licence:** CubiCasa5k is CC BY-NC-SA 4.0 (non-commercial). Fine for this experiment; check the terms before using a model trained on it in a paid product.

**Honest notes**
- The GPU code path was written on a laptop with no NVIDIA GPU, so it has **not been run on a GPU yet**. If a cell fails, copy the error back to Claude.
- The checkpoint is chosen on the *validation* split. Run the test cell once; if you re-run it while tuning, the test number stops being trustworthy.
- Colab sessions can disconnect. Training is ~25 epochs; the best checkpoint so far is saved to `unet.pt` after each improving epoch, so download it if a session is about to end.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
# Install the project straight from GitHub (PyTorch is already installed on Colab).
!pip -q install "git+https://github.com/Tgedin/buildable-land-map.git#subdirectory=plan-to-cost" opencv-python-headless scikit-image

## FAST PATH (recommended): upload the ready-made cache instead of downloading 5.4 GB

If the CubiCasa5k download is slow, skip the cells marked **A** and run the next cell instead.
`cache256_trainval.zip` (~80 MB, made on the laptop) holds the decoded training and validation
plans at 256 px. Upload it when asked. It only works with `SIZE = 256`.

The 400 test plans are **not** in it: score the model on your laptop afterwards (`unet.pt` is
tiny), so the test split is still used exactly once.

In [ ]:
SIZE = 256
from google.colab import files
files.upload()   # choose cache256_trainval.zip
!mkdir -p cache && unzip -o -q cache256_trainval.zip -d cache && ls -la cache

### A. Slow path: download and decode the full dataset (skip if you used the fast path)

In [ ]:
# Download CubiCasa5k (5.4 GB) and verify it. The published checksum is from Zenodo record 2613548.
!curl -L --retry 10 --retry-all-errors --retry-delay 5 -o cubicasa5k.zip "https://zenodo.org/records/2613548/files/cubicasa5k.zip?download=1"
!echo "0ce0b203d1e3c125b51087b219bd23b9  cubicasa5k.zip" | md5sum -c -
!unzip -q cubicasa5k.zip -d data && ls data/cubicasa5k | head

In [ ]:
# Decode every plan once into small arrays, so training never touches PNG/SVG again.
SIZE = 256   # try 384 or 512 on a GPU; then use the same value in the evaluation cell
!python -m plan2cost.learn.data --root data/cubicasa5k --out cache --size {SIZE}

In [ ]:
# Train. BASE is the network width (16 = 1.9M parameters, the laptop run); try 32 on a GPU.
BASE = 16
!python -u -m plan2cost.learn.train --cache cache --out unet.pt --epochs 25 --batch 16 --base {BASE} --lr 3e-3 --device cuda

In [ ]:
# Only works on the slow path (needs the raw dataset). On the fast path, skip this and
# score the model on your laptop: python -m plan2cost.eval.cubicasa --split test --model unet.pt
# Score on the 400 TEST plans, once. Same metrics and frame as the classical baseline.
!python -m plan2cost.eval.cubicasa --root data/cubicasa5k --split test --model unet.pt --model-size {SIZE} --out test_results.json

In [ ]:
from google.colab import files
files.download("unet.pt")
files.download("test_results.json")